# Long-Term Marine Environmental Monitoring Data Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² Long-Term Marine Environmental Monitoring Data from Estuaries and Coasts of the Basque Country (1995–2014) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, referencing the FAIR² standardized metadata and data.


In [ ]:
# Ensure the required library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL for FAIR²
croissant_url = 'https://sen.science/doi/10.71728/senscience.p1jx-e3n4/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"License: {meta.license}")
print(f"Version: {meta.version}")
print(f"Identifier: {meta.identifier}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

To interact with the dataset, we will use record set and field `@id` fields. Here, we list all record sets and their associated field `@id`s.

In [ ]:
# Print all available record set @ids and their fields
record_sets = [rs['@id'] for rs in dataset.record_sets]

print("Available Record Sets (@id fields):")
for rs in dataset.record_sets:
    print(f"  - {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("    Fields:")
    for f in fields:
        if isinstance(f, str):
            print(f"      - {f}")
        elif isinstance(f, dict):
            print(f"      - {f.get('@id', str(f))}")

if len(record_sets) == 0:
    print("(No record sets listed in top-level metadata. This dataset may use indirect references. See Example below for data iteration.")

## 3. Data Extraction
Load data from each record set listed above into a DataFrame using their `@id`.

If the dataset organizes data by record sets, we loop through them. If not (if record_sets is empty), we attempt to infer possible record set `@id`s from the data resource itself.

In [ ]:
# If record sets are defined:
dataframes = {}

if len(record_sets):
    ids_to_iterate = record_sets
else:
    # If not, try to infer by accessing the dataset's resources using the mlcroissant index
    # Using mlcroissant.dataset._schema['recordSet'] if available -- else try direct iteration
    schema_record_sets = getattr(dataset, 'record_sets', [])
    ids_to_iterate = [rs['@id'] for rs in schema_record_sets] if schema_record_sets else []
    
    if not ids_to_iterate:
        # Attempt to list record sets programmatically (fallback, may not always work)
        print("No record sets defined in metadata; trying to list top-level data directly...")
        # Try to load data directly
        try:
            records = list(dataset.records())
            if records:
                df = pd.DataFrame(records)
                dataframes['direct_records'] = df
                print("Loaded data with default record set method.")
                print("Columns:", df.columns.tolist())
                display(df.head())
        except Exception as err:
            print(f"Unable to extract records: {err}")
        ids_to_iterate = []
        
for recset_id in ids_to_iterate:
    print(f"\nExtracting data for Record Set: {recset_id}")
    try:
        recs = list(dataset.records(record_set=recset_id))
        df = pd.DataFrame(recs)
        dataframes[recset_id] = df
        print("Columns:", df.columns.tolist())
        display(df.head())
    except Exception as err:
        print(f"  Could not load data for {recset_id}: {err}")

## 4. Exploratory Data Analysis (EDA)
Apply standard data processing, including filtering records, normalizing numeric fields, and grouping by key attributes.

**Note:** Replace `<record_set_id>`, `<numeric_field_id>`, and `<group_field_id>` below with those discovered in the record sets above. For demonstration, we attempt to infer these from columns.

In [ ]:
import numpy as np

# Choose which record set and field to use for EDA
if dataframes:
    # Pick the first available record set for analysis
    main_record_set = list(dataframes.keys())[0]
    df = dataframes[main_record_set]
else:
    raise ValueError("No DataFrames loaded!")

print(f"Analyzing record set: {main_record_set}")

# Try to pick a numeric field (float or int column)
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()

if not numeric_columns:
    print("No numeric fields found in this record set. Please select another record set or field.")
else:
    # Select the first numeric column
    numeric_field_id = numeric_columns[0]
    print(f"Using numeric field: {numeric_field_id}")

    # Pick threshold as 10th percentile or a fixed value
    threshold = float(df[numeric_field_id].quantile(0.1)) if not df[numeric_field_id].isnull().all() else 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Choose a group field (categorical/text column)
    candidate_group_fields = df.select_dtypes(include='object').columns.tolist()
    group_field = candidate_group_fields[0] if candidate_group_fields else None

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field}:")
        display(grouped_df.head())
    else:
        print("No available categorical field for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields using `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not numeric_columns:
    print("No numeric data for visualization.")
else:
    # Histogram of the chosen numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

    if group_field:
        # Boxplot by group
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field}')
        plt.xticks(rotation=60, ha='right')
        plt.tight_layout()
        plt.show()


## 6. Conclusion
In this notebook, we:
- Loaded the FAIR² dataset metadata using its Croissant schema and `mlcroissant`.
- Explored the available record sets and fields by their `@id` identifiers.
- Loaded records into pandas DataFrames for further analysis.
- Performed basic EDA: filtering, normalization, and grouping by categorical identifiers, referencing all entities by their `@id`.
- Visualized numeric data distributions and group trends.

This approach supports fully reproducible, standards-based ecological data workflows.